# 🔋 Battery SoH Estimation — Week 4: SPM + Transformer

**WIDSS — Windowed Intelligent Drive-cycle State eStimation**

This notebook covers:
1. Single Particle Model (SPM) — physics-derived features
2. Cycle-aging simulation → State-of-Health (SoH) labels
3. Transformer architecture for SoC estimation
4. Physics-informed loss function
5. LSTM vs Transformer side-by-side comparison
6. SoH degradation curve fitting


## 0. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from widss.simulation import BatterySimulationConfig, build_dataset
from widss.dataset import build_sequences
from widss.evaluation import rmse, mae, mape

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
SEED = 42
rng  = np.random.default_rng(SEED)
print('✅ Ready')

---
## 1. Single Particle Model (SPM) — Physics Features

The SPM represents each electrode as a single spherical particle. It generates
physically meaningful features beyond raw current/voltage:
- **Lithium surface concentration** (proxy for SoC at particle surface)
- **Overpotential η** (reaction driving force)
- **Solid-phase concentration gradient**

We implement a simplified discrete-time SPM suitable for educational use.

In [ ]:
class SimpleSPM:
    """
    Simplified Single Particle Model (SPM) for a Li-ion cell.

    Models lithium concentration at the surface of a spherical particle
    using a 1st-order approximation of solid diffusion.

    Reference: Moura et al., IEEE Trans. Control Syst. Technol., 2017.
    """
    def __init__(self,
                 # Negative electrode (anode)
                 Ds_neg: float = 3.9e-14,   # Solid diffusivity (m²/s)
                 R_neg:  float = 12.5e-6,   # Particle radius (m)
                 eps_neg: float = 0.3,       # Volume fraction
                 L_neg:  float = 100e-6,    # Electrode thickness (m)
                 # Positive electrode (cathode)
                 Ds_pos: float = 1.0e-13,
                 R_pos:  float = 10.0e-6,
                 eps_pos: float = 0.3,
                 L_pos:  float = 80e-6,
                 # Cell parameters
                 F: float = 96485.0,        # Faraday's constant
                 A: float = 1.0,            # Electrode area (m²)
                 c_s_neg_max: float = 30555.0,  # Max Li concentration (mol/m³)
                 c_s_pos_max: float = 51554.0,
                 soc_init: float = 0.95,
                 dt: float = 1.0):

        self.F = F; self.A = A; self.dt = dt
        self.Ds_neg = Ds_neg; self.R_neg = R_neg; self.eps_neg = eps_neg; self.L_neg = L_neg
        self.Ds_pos = Ds_pos; self.R_pos = R_pos; self.eps_pos = eps_pos; self.L_pos = L_pos
        self.c_s_neg_max = c_s_neg_max
        self.c_s_pos_max = c_s_pos_max

        # Initial concentrations (stoichiometry mapped from SoC)
        theta_neg_0 = 0.8 + soc_init * (0.8 - 0.8)  # simplified
        theta_pos_0 = 1.0 - soc_init * 0.5
        self.c_neg = theta_neg_0 * c_s_neg_max
        self.c_pos = theta_pos_0 * c_s_pos_max

        # Diffusion time constants
        self.tau_neg = self.R_neg**2 / (15 * self.Ds_neg)
        self.tau_pos = self.R_pos**2 / (15 * self.Ds_pos)

        # Interfacial area density
        self.a_neg = 3 * eps_neg / R_neg
        self.a_pos = 3 * eps_pos / R_pos

    def step(self, current_a: float) -> dict:
        """Advance one timestep; current_a > 0 means discharging."""
        I = current_a

        # Pore-wall flux (mol/m² s)
        j_neg = I / (self.F * self.a_neg * self.L_neg * self.A)
        j_pos = -I / (self.F * self.a_pos * self.L_pos * self.A)

        # Surface concentration update (1st-order ODE approximation)
        dc_neg = -3 * j_neg / self.R_neg
        dc_pos = -3 * j_pos / self.R_pos
        self.c_neg += dc_neg * self.dt
        self.c_pos += dc_pos * self.dt

        # Clip to physical limits
        self.c_neg = np.clip(self.c_neg, 0, self.c_s_neg_max)
        self.c_pos = np.clip(self.c_pos, 0, self.c_s_pos_max)

        theta_neg = self.c_neg / self.c_s_neg_max
        theta_pos = self.c_pos / self.c_s_pos_max

        # Simple OCP approximation (polynomial fit)
        ocp_neg = 0.194 + 1.5 * np.exp(-120 * theta_neg) - 0.045 * np.tanh(15 * (theta_neg - 0.42))
        ocp_pos = 4.19829 + 0.0565661 * np.tanh(-14.5546 * theta_pos + 8.60) - 0.0275479 * \
                  (1 / (0.998432 - theta_pos)**0.492465 - 1.90111) - 0.157123 * np.exp(-0.04738 * theta_pos**8)

        # Overpotentials
        eta_neg = j_neg / (1e-4 * (theta_neg * (1 - theta_neg))**0.5 + 1e-10)
        eta_pos = j_pos / (1e-4 * (theta_pos * (1 - theta_pos))**0.5 + 1e-10)

        return {
            'theta_neg': theta_neg,
            'theta_pos': theta_pos,
            'ocp_neg': ocp_neg,
            'ocp_pos': ocp_pos,
            'eta_neg': eta_neg,
            'eta_pos': eta_pos,
            'j_neg': j_neg,
            'j_pos': j_pos
        }

print('SimpleSPM class defined ✅')

In [ ]:
# ── Generate dataset with SPM features ───────────────────────────────────────
cfg = BatterySimulationConfig(capacity_ah=60.0, soc_init=0.95, dt_s=1.0)
df_base = build_dataset(duration_s=7200, config=cfg, seed=SEED)

spm = SimpleSPM(soc_init=0.95, dt=1.0)
spm_records = [spm.step(I) for I in df_base['current_a'].values]
df_spm = pd.DataFrame(spm_records)

df_phys = pd.concat([df_base.reset_index(drop=True), df_spm], axis=1)
print(f'Dataset with SPM features: {df_phys.shape}')
print(f'New columns: {list(df_spm.columns)}')
df_phys.head()

In [ ]:
# ── SPM features vs SoC ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
axes = axes.flatten()

spm_cols = ['theta_neg', 'theta_pos', 'ocp_neg', 'ocp_pos', 'eta_neg', 'eta_pos']
titles   = ['θ_neg (anode stoichiometry)', 'θ_pos (cathode stoichiometry)',
            'OCP_neg (V)', 'OCP_pos (V)', 'η_neg (overpotential)', 'η_pos (overpotential)']

for ax, col, title in zip(axes, spm_cols, titles):
    ax.scatter(df_phys['soc'], df_phys[col], s=1, alpha=0.2, color='steelblue')
    ax.set_xlabel('SoC'); ax.set_ylabel(col)
    ax.set_title(title)

plt.suptitle('SPM Physics Features vs State of Charge', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## 2. Cycle-Aging Simulation → SoH Labels

In [ ]:
def simulate_aging(n_cycles: int = 500,
                   capacity_init_ah: float = 60.0,
                   resistance_init_ohm: float = 0.02,
                   alpha: float = 0.00015,   # capacity fade rate per cycle
                   beta:  float = 0.0001,    # resistance growth rate per cycle
                   noise_std: float = 0.001,
                   seed: int = 42) -> pd.DataFrame:
    """Simulate battery aging over many cycles (SEI growth model)."""
    rng = np.random.default_rng(seed)
    cycles = np.arange(1, n_cycles + 1)

    # Capacity fade: Q(n) = Q0 * exp(-alpha * n)  + noise
    capacity = capacity_init_ah * np.exp(-alpha * cycles) \
               + rng.normal(0, noise_std, n_cycles)

    # Resistance growth: R(n) = R0 * (1 + beta * sqrt(n)) + noise
    resistance = resistance_init_ohm * (1 + beta * np.sqrt(cycles)) \
                 + rng.normal(0, noise_std * 0.001, n_cycles)

    soh = capacity / capacity_init_ah  # SoH definition: capacity relative to nominal

    return pd.DataFrame({
        'cycle'     : cycles,
        'capacity_ah': capacity,
        'resistance_ohm': resistance,
        'soh'       : soh
    })

df_aging = simulate_aging(n_cycles=1000)
print(f'Aging data: {df_aging.shape}')
print(f'SoH range: [{df_aging.soh.min():.3f}, {df_aging.soh.max():.3f}]')
df_aging.head()

In [ ]:
# ── Degradation curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df_aging['cycle'], df_aging['capacity_ah'], color='steelblue', lw=1.2)
axes[0].axhline(0.8 * 60, color='r', ls='--', lw=0.8, label='80% capacity (EoL)')
axes[0].set_title('Capacity Fade'); axes[0].set_xlabel('Cycle')
axes[0].set_ylabel('Capacity (Ah)'); axes[0].legend(fontsize=9)

axes[1].plot(df_aging['cycle'], df_aging['resistance_ohm'] * 1000, color='darkorange', lw=1.2)
axes[1].set_title('Resistance Growth'); axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Internal Resistance (mΩ)')

axes[2].plot(df_aging['cycle'], df_aging['soh'] * 100, color='green', lw=1.2)
axes[2].axhline(80, color='r', ls='--', lw=0.8, label='80% SoH = EoL')
axes[2].set_title('State of Health (SoH)'); axes[2].set_xlabel('Cycle')
axes[2].set_ylabel('SoH (%)'); axes[2].legend(fontsize=9)

# End-of-Life cycle
eol_cycle = df_aging[df_aging['soh'] <= 0.8]['cycle'].min()
axes[2].axvline(eol_cycle, color='purple', ls=':', lw=1, label=f'EoL @ cycle {eol_cycle}')
axes[2].legend(fontsize=9)

plt.suptitle('Battery Aging Simulation (SEI Growth Model)', fontsize=13)
plt.tight_layout(); plt.show()
print(f'End-of-Life reached at cycle: {eol_cycle}')

---
## 3. SoH Prediction — Regression on Aging Data

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

# Features: cycle number, resistance (measured)
X_age = df_aging[['cycle', 'resistance_ohm']].values
y_age = df_aging['soh'].values

split_age = int(0.8 * len(df_aging))
X_tr_a, X_te_a = X_age[:split_age], X_age[split_age:]
y_tr_a, y_te_a = y_age[:split_age], y_age[split_age:]

sc_a = StandardScaler()
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=SEED)
gbr.fit(sc_a.fit_transform(X_tr_a), y_tr_a)
y_soh_pred = gbr.predict(sc_a.transform(X_te_a))

print('── GBR SoH Prediction ─────────────────────────────')
print(f'RMSE : {rmse(y_te_a, y_soh_pred):.5f}')
print(f'MAE  : {mae(y_te_a, y_soh_pred):.5f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_aging['cycle'].values[split_age:], y_te_a, 'k-', lw=1.5, label='True SoH')
ax.plot(df_aging['cycle'].values[split_age:], y_soh_pred, 'r--', lw=1.2, label='GBR Predicted')
ax.set_xlabel('Cycle'); ax.set_ylabel('SoH')
ax.set_title('SoH Prediction with Gradient Boosting')
ax.legend()
plt.tight_layout(); plt.show()

---
## 4. Transformer Architecture for SoC Estimation

Transformers use self-attention to model long-range dependencies in sequences —
potentially capturing charge patterns that span hundreds of timesteps.

In [ ]:
from widss.model import tensorflow_available

WINDOW_SIZE = 60

if not tensorflow_available():
    print('⚠️  TensorFlow not installed — skipping Transformer section.')
else:
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    # ── Transformer block ─────────────────────────────────────────────────────
    class TransformerBlock(layers.Layer):
        """Single Transformer encoder block: Multi-Head Attention + FFN."""

        def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
            super().__init__(**kwargs)
            self.att = layers.MultiHeadAttention(
                num_heads=num_heads, key_dim=embed_dim // num_heads)
            self.ffn = tf.keras.Sequential([
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim)
            ])
            self.ln1 = layers.LayerNormalization(epsilon=1e-6)
            self.ln2 = layers.LayerNormalization(epsilon=1e-6)
            self.drop1 = layers.Dropout(dropout_rate)
            self.drop2 = layers.Dropout(dropout_rate)

        def call(self, x, training=False):
            attn  = self.att(x, x, training=training)
            attn  = self.drop1(attn, training=training)
            out1  = self.ln1(x + attn)
            ffn   = self.ffn(out1)
            ffn   = self.drop2(ffn, training=training)
            return self.ln2(out1 + ffn)

    def build_transformer_soc(
        window_size, feature_count, embed_dim=32,
        num_heads=4, ff_dim=64, num_blocks=2,
        dropout=0.1, lr=1e-3
    ):
        inp = layers.Input(shape=(window_size, feature_count))

        # Linear projection to embed_dim
        x = layers.Dense(embed_dim)(inp)

        # Positional encoding (learnable)
        positions = tf.range(start=0, limit=window_size, delta=1)
        pos_emb   = layers.Embedding(input_dim=window_size, output_dim=embed_dim)(positions)
        x = x + pos_emb  # broadcast add

        # Stack Transformer blocks
        for _ in range(num_blocks):
            x = TransformerBlock(embed_dim, num_heads, ff_dim, dropout)(x)

        # Pool over sequence dimension
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(32, activation='relu')(x)
        x = layers.Dropout(dropout)(x)
        out = layers.Dense(1, activation='sigmoid')(x)  # SoC ∈ [0,1]

        model = Model(inp, out, name='SoC_Transformer')
        model.compile(
            optimizer=tf.keras.optimizers.Adam(lr),
            loss='mse',
            metrics=['mae']
        )
        return model

    print('Transformer architecture defined ✅')

In [ ]:
if tensorflow_available():
    # Use SPM-augmented features: voltage, current, theta_neg, theta_pos, eta_neg, eta_pos
    SPM_FEAT_COLS = ['voltage_v', 'current_a', 'theta_neg', 'theta_pos', 'eta_neg', 'eta_pos']

    def build_spm_sequences(df_full, feature_cols, window_size=60):
        data = df_full[feature_cols].values.astype('float32')
        target = df_full['soc'].values.astype('float32')
        X = np.stack([data[i:i+window_size] for i in range(len(data)-window_size)], axis=0)
        y = target[window_size:]
        return X, y

    # Normalize SPM features
    from sklearn.preprocessing import StandardScaler as SC
    sc_spm = SC()
    df_phys_norm = df_phys.copy()
    df_phys_norm[SPM_FEAT_COLS] = sc_spm.fit_transform(df_phys[SPM_FEAT_COLS])

    X_spm, y_spm = build_spm_sequences(df_phys_norm, SPM_FEAT_COLS, WINDOW_SIZE)
    split_t = int(0.8 * len(X_spm))
    X_tr_t, X_te_t = X_spm[:split_t], X_spm[split_t:]
    y_tr_t, y_te_t = y_spm[:split_t], y_spm[split_t:]
    print(f'SPM sequences — Train: {X_tr_t.shape}  |  Test: {X_te_t.shape}')

In [ ]:
if tensorflow_available():
    tf.random.set_seed(SEED)
    transformer = build_transformer_soc(
        window_size=WINDOW_SIZE,
        feature_count=len(SPM_FEAT_COLS),
        embed_dim=32, num_heads=4, ff_dim=64, num_blocks=2, dropout=0.1
    )
    transformer.summary()

In [ ]:
if tensorflow_available():
    callbacks_t = [
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ]

    history_t = transformer.fit(
        X_tr_t, y_tr_t,
        validation_split=0.15,
        epochs=30,
        batch_size=64,
        callbacks=callbacks_t,
        verbose=1
    )

In [ ]:
if tensorflow_available():
    y_trans_pred = np.clip(transformer.predict(X_te_t, verbose=0).flatten(), 0, 1)
    print('── Transformer Results ───────────────────────────────')
    print(f'RMSE : {rmse(y_te_t, y_trans_pred):.4f}')
    print(f'MAE  : {mae(y_te_t, y_trans_pred):.4f}')
    print(f'MAPE : {mape(y_te_t, y_trans_pred):.2f} %')

---
## 5. LSTM vs Transformer — Side-by-Side Comparison

In [ ]:
if tensorflow_available():
    from widss.model import build_lstm_soc_model

    # Train baseline LSTM on the same SPM-feature dataset
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dropout, Dense, BatchNormalization

    lstm_w4 = Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(SPM_FEAT_COLS))),
        Dropout(0.2),
        LSTM(32),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    lstm_w4.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])

    lstm_w4.fit(
        X_tr_t, y_tr_t,
        validation_split=0.15, epochs=20, batch_size=64,
        callbacks=callbacks_t, verbose=0
    )

    y_lstm_w4 = np.clip(lstm_w4.predict(X_te_t, verbose=0).flatten(), 0, 1)

    # ── Comparison table ──────────────────────────────────────────────────────
    compare = pd.DataFrame({
        'LSTM (Stacked, SPM features)': {
            'RMSE': rmse(y_te_t, y_lstm_w4),
            'MAE' : mae(y_te_t, y_lstm_w4),
            'MAPE': mape(y_te_t, y_lstm_w4)
        },
        'Transformer (SPM features)': {
            'RMSE': rmse(y_te_t, y_trans_pred),
            'MAE' : mae(y_te_t, y_trans_pred),
            'MAPE': mape(y_te_t, y_trans_pred)
        }
    }).T.round(4)

    print('Model Comparison (Test Set with SPM Features):')
    print(compare)

In [ ]:
if tensorflow_available():
    # ── Prediction plots ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    steps = np.arange(min(800, len(y_te_t)))

    axes[0].plot(steps, y_te_t[:800], 'k-', lw=1.5, label='True SoC')
    axes[0].plot(steps, y_lstm_w4[:800], 'b--', lw=1.2,
                 label=f'LSTM  (RMSE={rmse(y_te_t, y_lstm_w4):.4f})')
    axes[0].set_ylabel('SoC'); axes[0].set_title('LSTM — SPM Features')
    axes[0].legend()

    axes[1].plot(steps, y_te_t[:800], 'k-', lw=1.5, label='True SoC')
    axes[1].plot(steps, y_trans_pred[:800], 'r--', lw=1.2,
                 label=f'Transformer (RMSE={rmse(y_te_t, y_trans_pred):.4f})')
    axes[1].set_xlabel('Timestep'); axes[1].set_ylabel('SoC')
    axes[1].set_title('Transformer — SPM Features')
    axes[1].legend()

    plt.suptitle('LSTM vs Transformer: SoC Prediction (first 800 test steps)', fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
if tensorflow_available():
    # ── Training curves comparison ─────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(history_t.history['loss'], 'r-', label='Transformer Train')
    axes[0].plot(history_t.history['val_loss'], 'r--', label='Transformer Val')
    axes[0].set_title('Training Loss (MSE)'); axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history_t.history['mae'], 'r-', label='Transformer Train')
    axes[1].plot(history_t.history['val_mae'], 'r--', label='Transformer Val')
    axes[1].set_title('MAE'); axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.suptitle('Transformer Training Curves', fontsize=13)
    plt.tight_layout(); plt.show()

---
## 6. SoH Degradation Curve Fitting

In [ ]:
from scipy.optimize import curve_fit

def soh_model(n, Q0, alpha):
    """Exponential degradation model: SoH(n) = Q0 * exp(-alpha * n)"""
    return Q0 * np.exp(-alpha * n)

cycles_arr = df_aging['cycle'].values
soh_arr    = df_aging['soh'].values

popt, pcov = curve_fit(soh_model, cycles_arr, soh_arr, p0=[1.0, 1e-4])
Q0_fit, alpha_fit = popt
soh_fit = soh_model(cycles_arr, *popt)

print(f'Fitted Q0   = {Q0_fit:.4f}')
print(f'Fitted alpha = {alpha_fit:.6f} per cycle')
eol_pred = int(-np.log(0.8 / Q0_fit) / alpha_fit)
print(f'Predicted EoL (SoH=0.8): cycle {eol_pred}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(cycles_arr[::5], soh_arr[::5], s=4, alpha=0.5,
           color='steelblue', label='Simulated SoH data')
ax.plot(cycles_arr, soh_fit, 'r-', lw=1.5, label=f'Exp fit: SoH = {Q0_fit:.3f}·exp(-{alpha_fit:.4e}·n)')
ax.axhline(0.8, color='k', ls='--', lw=0.8, label='80% EoL threshold')
ax.axvline(eol_pred, color='purple', ls=':', lw=1.0, label=f'Predicted EoL @ cycle {eol_pred}')
ax.set_xlabel('Cycle'); ax.set_ylabel('SoH')
ax.set_title('SoH Degradation Curve Fitting (Exponential Model)')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## 7. Summary & Conclusions

In [ ]:
print('Week 4 Summary — SPM + Transformer')
print('='*60)
print()
print('1. SPM Physics Features')
print('   • Added θ_neg, θ_pos (stoichiometry), η_neg, η_pos (overpotential)')
print('   • These features have stronger physical correlation with SoC')
print()
print('2. Cycle-Aging Simulation (SoH)')
print('   • Exponential capacity fade model (SEI growth analogy)')
print(f'   • Simulated EoL (SoH ≤ 80%) at cycle {eol_pred}')
print()
print('3. Transformer Architecture')
print('   • Multi-Head Self-Attention over 60-step windows')
print('   • Learnable positional embeddings')
print()
if tensorflow_available():
    print('4. Model Comparison (SPM features):')
    print(compare.to_string())
print()
print('Next Steps (Roadmap Phase 3):')
print('  • REST inference API (FastAPI)')
print('  • ONNX export for embedded/MCU deployment')
print('  • Uncertainty quantification (MC Dropout / Bayesian LSTM)')
print('  • Real-world dataset integration (CALCE, NASA PCoE)')